In [1]:
import os

# Dữ liệu thực nghiệm: file du_lieu.json đặt cùng thư mục với notebook
DATA_FILE = "du_lieu.json"

assert os.path.exists(DATA_FILE), f"Không tìm thấy {DATA_FILE}. Hãy đặt file cùng thư mục với notebook."
print("Dùng file dữ liệu:", os.path.abspath(DATA_FILE))


Dùng file dữ liệu: c:\Users\Admin\Downloads\du_lieu.json


In [2]:
import json

with open(DATA_FILE, encoding="utf-8") as f:
    data = json.load(f)

print(type(data), len(data))
first = data[0] if isinstance(data, list) else list(data.values())[0]
print(first.keys())
print(first)


<class 'list'> 427
dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list'])
{'source_file': 'Benchmark Questions Verification V2.ipynb', 'task_id': 2, 'prompt': 'Write a function to find the shared elements from the given two lists.', 'code': 'def similar_elements(test_tup1, test_tup2):\n  res = tuple(set(test_tup1) & set(test_tup2))\n  return (res) ', 'test_imports': [], 'test_list': ['assert set(similar_elements((3, 4, 5, 6),(5, 7, 4, 10))) == set((4, 5))', 'assert set(similar_elements((1, 2, 3, 4),(5, 4, 3, 7))) == set((3, 4))', 'assert set(similar_elements((11, 12, 14, 13),(17, 15, 14, 13))) == set((13, 14))']}


In [3]:
items = data if isinstance(data, list) else list(data.values())

def pick(d, names):
    for n in names:
        if n in d:
            return d[n]
    raise KeyError(f"Không thấy trường nào trong {names}; các trường có: {list(d.keys())}")

with open("train.jsonl", "w", encoding="utf-8") as f:
    for d in items:
        req = pick(d, ["prompt", "text", "description"])
        code = pick(d, ["code", "solution"])
        f.write(json.dumps({"requirement": req, "code": code}, ensure_ascii=False) + "\n")

print("Đã tạo train.jsonl với", len(items), "mẫu")

Đã tạo train.jsonl với 427 mẫu


In [4]:
import json, os, random

with open(DATA_FILE, encoding="utf-8") as f:
    raw = json.load(f)

items = raw if isinstance(raw, list) else list(raw.values())

def pick(d, names):
    for n in names:
        if n in d:
            return d[n]
    raise KeyError(f"Không thấy trường nào trong {names}; các trường có: {list(d.keys())}")

rows = []
for d in items:
    rows.append({
        "requirement": pick(d, ["prompt", "text", "description"]),
        "code": pick(d, ["code", "solution"]),
        "tests": d.get("test_list", []),
    })

# Chia train/test để đánh giá không bị rò rỉ dữ liệu
random.seed(0)
random.shuffle(rows)
n_test = 100
test_rows, train_rows = rows[:n_test], rows[n_test:]

# Chạy thử nhanh: giảm số mẫu train, sau đó tăng dần (đặt None để dùng tất cả)
N_TRAIN = 100
if N_TRAIN:
    train_rows = train_rows[:N_TRAIN]

with open("train.jsonl", "w", encoding="utf-8") as f:
    for r in train_rows:
        f.write(json.dumps({"requirement": r["requirement"], "code": r["code"]},
                           ensure_ascii=False) + "\n")

with open("test.jsonl", "w", encoding="utf-8") as f:
    for r in test_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Đã tạo train.jsonl ({len(train_rows)} mẫu) và test.jsonl ({len(test_rows)} mẫu)")
print("Ví dụ:", train_rows[0]["requirement"])

Đã tạo train.jsonl (100 mẫu) và test.jsonl (100 mẫu)
Ví dụ: Write a function to find the maximum sum possible by using the given equation f(n) = max( (f(n/2) + f(n/3) + f(n/4) + f(n/5)), n).


In [5]:
from __future__ import annotations

import json
import math
import random
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, List, Sequence

# ---------------------------------------------------------------------------
# Cấu hình
# ---------------------------------------------------------------------------

DEFAULT_LLM = "Salesforce/codegen-350M-mono"   # đổi sang CodeGen-16B, CodeLlama-7B/34B... nếu đủ GPU
DEFAULT_ENCODER = "microsoft/graphcodebert-base"  # Transformer 12 lớp, ~125M tham số


@dataclass
class Example:
    requirement: str
    code: str


def load_jsonl(path: str) -> List[Example]:
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                d = json.loads(line)
                out.append(Example(d["requirement"], d["code"]))
    return out


def build_prompt(demos: Sequence[Example], requirement: str) -> str:
    """Ghép prompt few-shot: các ví dụ minh họa + yêu cầu mới."""
    parts = []
    for d in demos:
        parts.append(f"# Requirement:\n# {d.requirement}\n# Code:\n{d.code}\n")
    parts.append(f"# Requirement:\n# {requirement}\n# Code:\n")
    return "\n".join(parts)


# ---------------------------------------------------------------------------
# BM25 (tự cài đặt, không cần thư viện ngoài) - dùng để lọc thô ở bước A
# ---------------------------------------------------------------------------

def tokenize(text: str) -> List[str]:
    return re.findall(r"\w+", text.lower())


class BM25:
    def __init__(self, docs: Sequence[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(tokenize(d)) for d in docs]
        self.len = [sum(c.values()) for c in self.tf]
        self.avg = sum(self.len) / max(len(docs), 1)
        df = Counter(t for c in self.tf for t in c)
        n = len(docs)
        self.idf = {t: math.log(1 + (n - v + 0.5) / (v + 0.5)) for t, v in df.items()}

    def scores(self, query: str) -> List[float]:
        q = tokenize(query)
        res = []
        for tf, ln in zip(self.tf, self.len):
            s = 0.0
            for t in q:
                if t in tf:
                    f = tf[t]
                    s += self.idf[t] * f * (self.k1 + 1) / (
                        f + self.k1 * (1 - self.b + self.b * ln / self.avg))
            res.append(s)
        return res

    def top_n(self, query: str, n: int, exclude: int | None = None) -> List[int]:
        sc = self.scores(query)
        order = sorted(range(len(sc)), key=lambda i: sc[i], reverse=True)
        return [i for i in order if i != exclude][:n]


# ---------------------------------------------------------------------------
# GIAI ĐOẠN A - Estimating Examples with LLMs
# ---------------------------------------------------------------------------

class LLMScorer:
    """Tính xác suất LLM sinh đúng ground-truth code khi cho trước 1 ví dụ minh họa."""

    def __init__(self, model_name: str = DEFAULT_LLM, device: str | None = None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_name)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype).to(self.device).eval()

    def score(self, demo: Example, target: Example) -> float:
        """Trung bình log-prob mỗi token của target.code, với demo làm ngữ cảnh.
        exp(giá trị này) = xác suất trung bình hình học trên mỗi token."""
        torch = self.torch
        prompt = build_prompt([demo], target.requirement)
        p_ids = self.tok(prompt, return_tensors="pt").input_ids[0]
        c_ids = self.tok(target.code, return_tensors="pt", add_special_tokens=False).input_ids[0]
        ids = torch.cat([p_ids, c_ids]).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(ids).logits[0, :-1]
        logp = torch.log_softmax(logits.float(), dim=-1)
        tgt = ids[0, 1:]
        tok_logp = logp.gather(1, tgt.unsqueeze(1)).squeeze(1)
        code_logp = tok_logp[len(p_ids) - 1:]          # chỉ tính phần code
        return code_logp.mean().item()


def estimate_examples(
    train: List[Example],
    score_fn: Callable[[Example, Example], float],
    n_candidates: int = 50,
    n_pos: int = 1,
    n_neg: int = 7,
    log_every: int = 10,
) -> List[Dict]:
    """Với mỗi mẫu huấn luyện (làm 'query'):
       1. BM25 lấy n_candidates ứng viên (bỏ chính nó)
       2. LLM chấm điểm từng ứng viên
       3. Top n_pos -> dương, n_neg thấp nhất -> âm
    """
    bm25 = BM25([e.requirement + " " + e.code for e in train])
    labels = []
    for qi, q in enumerate(train):
        cands = bm25.top_n(q.requirement, n_candidates, exclude=qi)
        scored = sorted(((score_fn(train[c], q), c) for c in cands), reverse=True)
        pos = [c for _, c in scored[:n_pos]]
        neg = [c for _, c in scored[-n_neg:] if c not in pos]
        labels.append({"query": qi, "pos": pos, "neg": neg})
        if (qi + 1) % log_every == 0:
            print(f"[A] đã gán nhãn {qi + 1}/{len(train)}")
    return labels


# ---------------------------------------------------------------------------
# GIAI ĐOẠN B - Training a Model-Aware Retriever (Contrastive Learning)
# ---------------------------------------------------------------------------

def example_text(e: Example) -> str:
    return e.requirement + "\n" + e.code


class Retriever:
    """Encoder dạng Transformer (mặc định GraphCodeBERT) -> vector, so khớp bằng cosine."""

    def __init__(self, path_or_name: str = DEFAULT_ENCODER, device: str | None = None):
        import torch
        from transformers import AutoModel, AutoTokenizer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(path_or_name)
        self.model = AutoModel.from_pretrained(path_or_name).to(self.device)

    def encode(self, texts: Sequence[str], max_len: int = 256, grad: bool = False):
        torch = self.torch
        batch = self.tok(list(texts), padding=True, truncation=True,
                         max_length=max_len, return_tensors="pt").to(self.device)
        ctx = torch.enable_grad() if grad else torch.no_grad()
        with ctx:
            out = self.model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1).float()
            emb = (out * mask).sum(1) / mask.sum(1)          # mean pooling
        return torch.nn.functional.normalize(emb, dim=-1)

    def train(self, train: List[Example], labels: List[Dict], epochs: int = 3,
              batch_size: int = 8, lr: float = 2e-5, temperature: float = 0.05):
        """InfoNCE: kéo query lại gần ví dụ dương, đẩy xa ví dụ âm (+ in-batch negatives)."""
        torch = self.torch
        data = [l for l in labels if l["pos"] and l["neg"]]
        opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.model.train()
        for ep in range(epochs):
            random.shuffle(data)
            total = 0.0
            for i in range(0, len(data), batch_size):
                chunk = data[i:i + batch_size]
                q = self.encode([train[l["query"]].requirement for l in chunk], grad=True)
                p = self.encode([example_text(train[random.choice(l["pos"])]) for l in chunk],
                                grad=True)
                # mỗi query có số âm khác nhau -> dùng n_neg nhỏ nhất của batch
                m = min(len(l["neg"]) for l in chunk)
                negs = [example_text(train[c]) for l in chunk for c in l["neg"][:m]]
                n = self.encode(negs, grad=True).view(len(chunk), m, -1)

                pos_logit = (q * p).sum(-1, keepdim=True)                 # (B,1)
                neg_logit = torch.einsum("bd,bmd->bm", q, n)              # (B,m) hard neg
                inbatch = q @ p.T                                         # (B,B)
                inbatch.fill_diagonal_(float("-inf"))                     # bỏ cặp dương chính nó
                logits = torch.cat([pos_logit, neg_logit, inbatch], 1) / temperature
                loss = torch.nn.functional.cross_entropy(
                    logits, torch.zeros(len(chunk), dtype=torch.long, device=self.device))

                opt.zero_grad()
                loss.backward()
                opt.step()
                total += loss.item()
            print(f"[B] epoch {ep + 1}/{epochs}  loss={total / max(1, math.ceil(len(data) / batch_size)):.4f}")
        self.model.eval()

    def save(self, path: str):
        Path(path).mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(path)
        self.tok.save_pretrained(path)


# ---------------------------------------------------------------------------
# GIAI ĐOẠN C - Generating Programs
# ---------------------------------------------------------------------------

class LAIL:
    def __init__(self, retriever: Retriever, pool: List[Example],
                 llm_name: str = DEFAULT_LLM, device: str | None = None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.retriever, self.pool = retriever, pool
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(llm_name)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_name, torch_dtype=dtype).to(self.device).eval()
        # mã hóa trước toàn bộ kho ví dụ (làm 1 lần)
        self.pool_emb = torch.cat([
            retriever.encode([example_text(e) for e in pool[i:i + 32]])
            for i in range(0, len(pool), 32)])

    def select(self, requirement: str, r: int = 4) -> List[Example]:
        q = self.retriever.encode([requirement])
        sims = (q @ self.pool_emb.T)[0]                       # cosine (đã chuẩn hóa)
        idx = sims.topk(min(r, len(self.pool))).indices.tolist()
        return [self.pool[i] for i in reversed(idx)]          # ví dụ giống nhất đặt sát cuối

    def generate(self, requirement: str, r: int = 4, max_new_tokens: int = 256) -> str:
        demos = self.select(requirement, r)
        prompt = build_prompt(demos, requirement)
        ids = self.tok(prompt, return_tensors="pt").input_ids.to(self.device)
        with self.torch.no_grad():
            out = self.llm.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=self.tok.pad_token_id)
        text = self.tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        return text.split("\n# Requirement:")[0].rstrip()     # cắt khi LLM bắt đầu ví dụ mới


In [6]:
import os, time, json, torch

torch.set_num_threads(os.cpu_count())

class FastScorer(LLMScorer):
    def __init__(self, *args, quantize: bool = False, **kwargs):
        super().__init__(*args, **kwargs)
        if quantize and self.device == "cpu":
            self.model = torch.ao.quantization.quantize_dynamic(
                self.model, {torch.nn.Linear}, dtype=torch.qint8)

    @torch.inference_mode()
    def score_pairs(self, pairs, max_prompt=192, max_code=64, batch_size=12):
        """pairs = [(demo, target), ...] -> list điểm (log-prob TB/token của target.code)."""
        pad = self.tok.pad_token_id
        items = []
        for k, (demo, target) in enumerate(pairs):
            c = self.tok(target.code, add_special_tokens=False).input_ids[:max_code]
            p = self.tok(build_prompt([demo], target.requirement)).input_ids[-max_prompt:]
            items.append((k, p, c))
        items.sort(key=lambda x: len(x[1]) + len(x[2]))   # giảm padding

        scores = [0.0] * len(pairs)
        can_slice = hasattr(self.model, "transformer") and hasattr(self.model, "lm_head")

        for s in range(0, len(items), batch_size):
            chunk = items[s:s + batch_size]
            L = max(len(p) + len(c) for _, p, c in chunk)
            ids = torch.full((len(chunk), L), pad, dtype=torch.long)
            mask = torch.zeros((len(chunk), L), dtype=torch.long)
            for i, (_, p, c) in enumerate(chunk):
                seq = p + c
                ids[i, :len(seq)] = torch.tensor(seq)
                mask[i, :len(seq)] = 1
            ids, mask = ids.to(self.device), mask.to(self.device)

            if can_slice:   # chỉ tính lm_head cho vị trí phần code
                h = self.model.transformer(input_ids=ids, attention_mask=mask).last_hidden_state
            else:
                logits = self.model(input_ids=ids, attention_mask=mask).logits

            for i, (k, p, c) in enumerate(chunk):
                if not c:
                    continue
                lp, lc = len(p), len(c)
                lg = self.model.lm_head(h[i, lp - 1: lp - 1 + lc]) if can_slice \
                     else logits[i, lp - 1: lp - 1 + lc]
                lg = torch.log_softmax(lg.float(), dim=-1)
                tgt = ids[i, lp: lp + lc]
                scores[k] = lg.gather(1, tgt.unsqueeze(1)).mean().item()
        return scores


def estimate_fast(train, scorer, n_candidates=3, n_pos=1, n_neg=2,
                  ckpt="labels_partial.json", q_per_batch=4):
    bm25 = BM25([e.requirement + " " + e.code for e in train])
    labels = []
    if os.path.exists(ckpt):
        with open(ckpt, encoding="utf-8") as f:
            labels = json.load(f)
        print(f"Tiếp tục từ mẫu {len(labels)}")
    start, t0 = len(labels), time.time()

    for s in range(start, len(train), q_per_batch):
        qis = list(range(s, min(s + q_per_batch, len(train))))
        cand_lists = [bm25.top_n(train[qi].requirement, n_candidates, exclude=qi) for qi in qis]
        pairs = [(train[c], train[qi]) for qi, cands in zip(qis, cand_lists) for c in cands]
        scores = scorer.score_pairs(pairs)

        k = 0
        for qi, cands in zip(qis, cand_lists):
            sc = scores[k:k + len(cands)]
            k += len(cands)
            scored = sorted(zip(sc, cands), reverse=True)
            pos = [c for _, c in scored[:n_pos]]
            neg = [c for _, c in scored[-n_neg:] if c not in pos]
            labels.append({"query": qi, "pos": pos, "neg": neg})

        with open(ckpt, "w", encoding="utf-8") as f:
            json.dump(labels, f, ensure_ascii=False)
        done = len(labels) - start
        eta = (time.time() - t0) / done * (len(train) - len(labels))
        print(f"[A] {len(labels)}/{len(train)}  còn khoảng {eta:.0f} giây")
    return labels


train = load_jsonl("train.jsonl")

if os.path.exists("labels.json"):
    with open("labels.json", encoding="utf-8") as f:
        labels = json.load(f)
    print("Đã có labels.json, bỏ qua bước A")
else:
    t0 = time.time()
    scorer = FastScorer(quantize=True)   # int8 trên CPU; đổi False nếu báo lỗi
    labels = estimate_fast(train, scorer, n_candidates=3, n_pos=1, n_neg=2, q_per_batch=4)
    with open("labels.json", "w", encoding="utf-8") as f:
        json.dump(labels, f, ensure_ascii=False, indent=1)
    del scorer
    print(f"Xong bước A sau {time.time() - t0:.0f} giây, đã lưu labels.json")

for l in labels[:3]:
    print(l)

Đã có labels.json, bỏ qua bước A
{'query': 0, 'pos': [3], 'neg': [9, 4, 5]}
{'query': 1, 'pos': [6], 'neg': [5, 4, 9]}
{'query': 2, 'pos': [6], 'neg': [5, 4, 9]}


In [7]:
import os, time, torch

torch.set_num_threads(os.cpu_count())   # dùng hết nhân CPU

if os.path.isdir("retriever_ckpt"):     # đã huấn luyện rồi thì nạp lại, không train nữa
    ret = Retriever("retriever_ckpt")
    print("Đã có retriever_ckpt, bỏ qua bước B")
else:
    t0 = time.time()
    ret = Retriever()   # microsoft/graphcodebert-base

    # Đóng băng embedding + 8 lớp đầu, chỉ huấn luyện 4 lớp cuối
    for p in ret.model.embeddings.parameters():
        p.requires_grad = False
    for layer in ret.model.encoder.layer[:8]:
        for p in layer.parameters():
            p.requires_grad = False

    ret.train(train, labels, epochs=2, batch_size=2)
    ret.save("retriever_ckpt")
    print(f"Xong bước B sau {time.time() - t0:.0f} giây, đã lưu retriever_ckpt")

c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:03<00:00, 64.79it/s] 


Đã có retriever_ckpt, bỏ qua bước B


In [8]:
lail = LAIL(ret, train)  # Dùng lại retriever đã huấn luyện

# Hiển thị toàn bộ yêu cầu từ data
options = {
    chr(65 + i): item.requirement
    for i, item in enumerate(train)
}

print("=== CHỌN YÊU CẦU SINH MÃ NGUỒN ===")

for key, requirement in options.items():
    print(f"{key}. {requirement}")

# Nhập lựa chọn
choice = input("\nNhập lựa chọn : ").strip().upper()

if choice in options:
    requirement = options[choice]

    print("\n=== Yêu cầu được chọn ===")
    print(requirement)

    print("\n=== Ví dụ được chọn ===")
    for d in lail.select(requirement, r=4):
        print("-", d.requirement)

    print("\n=== Code sinh ra ===")
    print(lail.generate(requirement, r=4))

else:
    print("Lựa chọn không hợp lệ! Vui lòng chọn lại.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 165/165 [00:27<00:00,  5.91it/s]
[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== CHỌN YÊU CẦU SINH MÃ NGUỒN ===
A. Write a function to find the maximum sum possible by using the given equation f(n) = max( (f(n/2) + f(n/3) + f(n/4) + f(n/5)), n).
B. Write a function which returns nth catalan number.
C. Write a function to find the nth hexagonal number.
D. Write a function that takes in a list and an integer L and splits the given list into two parts where the length of the first part of the list is L, and returns the resulting lists in a tuple.
E. Write a python function to find the sum of even factors of a number.
F. Write a python function to find the sum of xor of all pairs of numbers in the given list.
G. Write a function to sum the length of the names of a given list of names after removing the names that start with a lowercase letter.
H. Write a python function to check whether the given two integers have opposite sign or not.
I. Write a function to caluclate the area of a tetrahedron.
J. Write a function to find the median of two sorted lists of same size

In [9]:
# ============================================================
# ĐÁNH GIÁ PASS@K - BẢN 50 BÀI (khoảng 1 giờ 20 phút trên CPU, TỰ LƯU TIẾN ĐỘ, chạy lại thì tiếp tục)
# Chạy sau khi đã có: lail (có select/tok/llm/device), build_prompt, và file test.jsonl
# Trong notebook: dán vào một ô, hoặc chạy:  %run -i pass_at_k_50bai.py
# ============================================================
import ast, json, math, os, re, subprocess, sys, tempfile, time

# ---------------- Cấu hình ----------------
TIME_BUDGET = 3 * 3600   # giây, giới hạn an toàn cho phần việc MỚI trong một lần chạy (3 giờ)
N_SAMPLES = 5            # số lời giải sinh cho mỗi bài (bản gốc: 10)
K_VALUES = [1, 5]        # các chỉ số pass@k cần báo cáo (k <= N_SAMPLES)
R_DEMOS = 3              # số ví dụ minh họa trong prompt (bản gốc: 4). Ít hơn = prompt ngắn = nhanh hơn
MAX_NEW_TOKENS = 100     # bản gốc: 256. Đủ cho hầu hết hàm MBPP; có dừng sớm nên thường ngắn hơn
TEMPERATURE, TOP_P = 0.8, 0.95
N_EVAL = 50              # số bài đánh giá (sai số chuẩn của pass@1 khoảng 6 điểm % nếu pass@1 ≈ 30%)
PROGRESS = "pass_at_k_progress.json"   # tiến độ lưu sau MỖI bài; chạy lại sẽ bỏ qua các bài đã xong
USE_ASSERT_HINT = True   # đưa 1 câu assert mẫu vào yêu cầu (cách chuẩn của MBPP, giúp mô hình biết tên hàm)
                         # Báo cáo phải ghi rõ nếu bật. Đặt False để giống hệt bản gốc.

with open("test.jsonl", "r", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]


def clean_generated_code(text):
    """Bỏ markdown fence, cắt phần mô hình bắt đầu ví dụ mới, bỏ dòng dở dang, chỉ giữ import/def/class/gán."""
    text = text.strip()
    text = re.sub(r"^```(?:python)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    src = text.split("\n# Requirement")[0].rstrip()
    lines = src.split("\n")
    while lines:
        try:
            tree = ast.parse("\n".join(lines))
            break
        except SyntaxError:
            lines.pop()
    else:
        return ""
    src = "\n".join(lines)
    keep = (ast.FunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom, ast.Assign)
    return "\n".join(ast.get_source_segment(src, n) for n in tree.body if isinstance(n, keep))


def make_prompt(demos, requirement, first_test):
    if USE_ASSERT_HINT:
        if "aug_demo" in globals():                       # có sẵn từ notebook đã sửa: ví dụ minh họa cũng kèm assert
            demos = [aug_demo(d) for d in demos]
        requirement = f"{requirement}\n# Your code should pass: {first_test}"
    return build_prompt(demos, requirement)


def sample_candidates(model, requirement, first_test, k):
    """Sinh k lời giải khác nhau bằng 1 lần gọi generate, dừng sớm khi cả k lời giải đã xong."""
    import torch
    from transformers import StoppingCriteria, StoppingCriteriaList

    demos = model.select(requirement, r=R_DEMOS)
    prompt = make_prompt(demos, requirement, first_test)
    inputs = model.tok(prompt, return_tensors="pt").to(model.device)
    start = inputs["input_ids"].shape[1]

    class Stop(StoppingCriteria):
        """Mỗi chuỗi dừng khi bắt đầu viết ví dụ mới hoặc lệnh thừa (không phí token)."""
        def __call__(self, input_ids, scores, **kw):
            return torch.tensor(
                [any(s in model.tok.decode(row[start:][-16:])
                     for s in ("# Requirement", "\nprint(", "\nassert ", "\n\n\n"))
                 for row in input_ids], device=input_ids.device)

    with torch.inference_mode():
        out = model.llm.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
            temperature=TEMPERATURE, top_p=TOP_P, num_return_sequences=k,
            pad_token_id=model.tok.pad_token_id,
            stopping_criteria=StoppingCriteriaList([Stop()]))
    return [clean_generated_code(model.tok.decode(o[start:], skip_special_tokens=True)) for o in out]


_check_cache = {}


def check_one_program(code, tests, setup_code="", timeout=5):
    """Chạy mã + test trong TIẾN TRÌNH RIÊNG có giới hạn thời gian (vòng lặp vô hạn không làm treo kernel)."""
    if not code.strip() or not tests:
        return False
    key = (code, tuple(tests))
    if key in _check_cache:                                # các lời giải trùng nhau chỉ chấm một lần
        return _check_cache[key]
    src = "import math, re, heapq, itertools, collections\n" + (setup_code + "\n" if setup_code else "") \
          + code + "\n" + "\n".join(tests)
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(src)
        name = f.name
    try:
        ok = subprocess.run([sys.executable, name], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        ok = False
    finally:
        os.remove(name)
    _check_cache[key] = ok
    return ok


def pass_at_k(n, c, k):
    """Ước lượng không chệch pass@k: n = số mẫu, c = số mẫu đúng."""
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)


def run():
    cfg = {"n_samples": N_SAMPLES, "r_demos": R_DEMOS, "max_new_tokens": MAX_NEW_TOKENS,
           "assert_hint": USE_ASSERT_HINT, "temperature": TEMPERATURE, "top_p": TOP_P}
    done = {}
    if os.path.exists(PROGRESS):
        saved = json.load(open(PROGRESS, encoding="utf-8"))
        if saved.get("config") == cfg:
            done = {r["task_id"]: r for r in saved["details"]}
            print(f"Tiếp tục: đã có {len(done)} bài từ lần chạy trước.")
        else:
            print("Cấu hình đã đổi so với lần chạy trước -> bỏ tiến độ cũ, chạy lại từ đầu.")

    total = min(N_EVAL, len(test_rows))
    all_results, new_times = [], []
    t_start = time.time()
    for idx, item in enumerate(test_rows[:total], start=1):
        if idx in done:
            all_results.append(done[idx])
            continue
        if time.time() - t_start > TIME_BUDGET:
            print(f"\nHết {TIME_BUDGET} giây cho lần chạy này, dừng ở bài {idx - 1}. Chạy lại để tiếp tục.")
            break
        t_problem = time.time()
        requirement = item["requirement"]
        tests = item.get("tests", [])
        setup_code = item.get("test_setup_code", "")

        candidates = sample_candidates(lail, requirement, tests[0] if tests else "", N_SAMPLES)
        c = sum(check_one_program(code, tests, setup_code) for code in candidates)
        n = len(candidates)
        ks = [k for k in K_VALUES if k <= n]
        all_results.append({"task_id": idx, "requirement": requirement, "n": n, "correct": c,
                            "pass_at_k": {f"pass@{k}": pass_at_k(n, c, k) for k in ks}})
        new_times.append(time.time() - t_problem)
        json.dump({"config": cfg, "details": all_results}, open(PROGRESS, "w", encoding="utf-8"),
                  ensure_ascii=False, indent=1)                      # lưu ngay sau mỗi bài
        remaining = (total - idx) * (sum(new_times) / len(new_times))
        print(f"[{idx}/{total}] Passed {c}/{n} | " + " | ".join(f"pass@{k}={pass_at_k(n, c, k):.3f}" for k in ks)
              + f" | {new_times[-1]:.0f}s/bài | còn khoảng {remaining / 60:.0f} phút", flush=True)

    summary = {}
    for k in K_VALUES:
        key = f"pass@{k}"
        vals = [r["pass_at_k"][key] for r in all_results if key in r["pass_at_k"]]
        if vals:
            summary[key] = sum(vals) / len(vals)

    print("\n========== KẾT QUẢ ĐÁNH GIÁ ==========")
    print(f"Số bài đã đánh giá: {len(all_results)} / {total}")
    print(f"Số mẫu sinh mỗi bài: {N_SAMPLES} | ví dụ minh họa: {R_DEMOS} | gợi ý assert: {USE_ASSERT_HINT}")
    for metric, value in summary.items():
        se = (value * (1 - value) / max(len(all_results), 1)) ** 0.5
        print(f"{metric}: {value:.4f} ({value * 100:.2f}%)  ± {se * 100:.1f} điểm % (sai số chuẩn xấp xỉ)")
    if new_times:
        print(f"Thời gian lần chạy này: {(time.time() - t_start) / 60:.1f} phút | trung bình {sum(new_times) / len(new_times):.0f} giây/bài")
    if len(all_results) < 30:
        print("Lưu ý: dưới 30 bài nên kết quả chỉ mang tính minh họa.")

    with open("pass_at_k_results.json", "w", encoding="utf-8") as f:
        json.dump({"summary": summary, "details": all_results, "config": cfg, "n_problems": len(all_results)},
                  f, ensure_ascii=False, indent=2)
    print("Đã lưu kết quả vào pass_at_k_results.json")


run()

Tiếp tục: đã có 50 bài từ lần chạy trước.

========== KẾT QUẢ ĐÁNH GIÁ ==========
Số bài đã đánh giá: 50 / 50
Số mẫu sinh mỗi bài: 5 | ví dụ minh họa: 3 | gợi ý assert: True
pass@1: 0.1400 (14.00%)  ± 4.9 điểm % (sai số chuẩn xấp xỉ)
pass@5: 0.3400 (34.00%)  ± 6.7 điểm % (sai số chuẩn xấp xỉ)
Đã lưu kết quả vào pass_at_k_results.json
